# House Price Prediction – End-to-End Machine Learning Project

This notebook demonstrates a complete regression workflow:

1. Import libraries
2. Load and inspect the dataset
3. Perform exploratory data analysis
4. Clean and prepare data
5. Split training and testing data
6. Build a preprocessing and regression pipeline
7. Train the model
8. Evaluate model performance
9. Compare actual and predicted prices
10. Make a new prediction
11. Save the trained model


## 1. Import Required Libraries

Run the installation command only when the libraries are not already installed.


In [ ]:
# Uncomment and run this line when packages are missing:
# !pip install pandas numpy matplotlib scikit-learn joblib


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)


## 2. Load the Dataset


In [ ]:
data = pd.read_csv("house_price_ml_dataset.csv")
data.head()


## 3. Inspect the Dataset


In [ ]:
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())


In [ ]:
data.info()


In [ ]:
data.describe(include="all").T


## 4. Check Missing Values and Duplicates


In [ ]:
print("Missing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())


In [ ]:
# Remove duplicate rows if any exist
data = data.drop_duplicates().copy()


## 5. Exploratory Data Analysis

The charts below help us understand price distribution and important relationships.


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(data["price_gbp"], bins=25, edgecolor="black")
plt.title("Distribution of House Prices")
plt.xlabel("Price in GBP")
plt.ylabel("Number of Properties")
plt.show()


In [ ]:
location_prices = (
    data.groupby("location")["price_gbp"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
location_prices.plot(kind="bar")
plt.title("Average House Price by Location")
plt.xlabel("Location")
plt.ylabel("Average Price in GBP")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data["area_sqft"], data["price_gbp"], alpha=0.7)
plt.title("Area vs House Price")
plt.xlabel("Area in Square Feet")
plt.ylabel("Price in GBP")
plt.show()


## 6. Define Features and Target

`price_gbp` is the value the model must predict.

`property_id` is only an identifier, so it is removed from the model features.


In [ ]:
X = data.drop(columns=["price_gbp", "property_id"])
y = data["price_gbp"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
X.head()


## 7. Identify Categorical and Numerical Columns


In [ ]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)


## 8. Split Training and Testing Data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 9. Create the Preprocessing and Model Pipeline

- `OneHotEncoder` converts text columns such as location and property type into numbers.
- `StandardScaler` scales numerical features.
- `LinearRegression` predicts the house price.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_columns
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

model


## 10. Train the Model


In [ ]:
model.fit(X_train, y_train)
print("Model training completed successfully.")


## 11. Make Predictions


In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": np.round(y_pred, 2),
    "Difference": np.round(y_test.values - y_pred, 2)
})

results.head(10)


## 12. Evaluate Model Performance


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: £{mae:,.2f}")
print(f"Root Mean Squared Error: £{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")


### Understanding the metrics

- **MAE:** Average prediction error in pounds.
- **RMSE:** Penalizes larger prediction errors.
- **R²:** Measures how much of the price variation is explained by the model. A value closer to 1 is better.


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)

minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())

plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.title("Actual Price vs Predicted Price")
plt.xlabel("Actual Price in GBP")
plt.ylabel("Predicted Price in GBP")
plt.show()


## 13. Predict the Price of a New Property


In [ ]:
new_property = pd.DataFrame({
    "location": ["Manchester"],
    "property_type": ["Semi-Detached"],
    "bedrooms": [3],
    "bathrooms": [2],
    "area_sqft": [1450],
    "age_years": [12],
    "distance_city_centre_km": [6.5],
    "has_garden": [1],
    "has_garage": [1],
    "nearby_school_rating": [4.2],
    "crime_rate_index": [24.0]
})

predicted_price = model.predict(new_property)[0]

print(f"Predicted house price: £{predicted_price:,.2f}")


## 14. Save the Trained Model


In [ ]:
joblib.dump(model, "house_price_model.pkl")
print("Model saved as house_price_model.pkl")


## 15. Load and Test the Saved Model


In [ ]:
loaded_model = joblib.load("house_price_model.pkl")
loaded_prediction = loaded_model.predict(new_property)[0]

print(f"Prediction from loaded model: £{loaded_prediction:,.2f}")


## Next Step

Run the included Streamlit application with:

```bash
streamlit run app.py
```

The app loads `house_price_model.pkl` and provides a form for predicting house prices.
